# Editing images - Fast!

A typical usage of "kernels" is to edit images. In this example, we will see how to use the `KernelAbstractions` library to edit images in a fast and efficient way.


#### Content
- [Introduction](#introduction)
- [Simple image kernel usage](#simple-image-kernels)
- [Large scale editing - comparison with CPU](#large-scale-editing---comparison-with-cpu)





This notebook was run on MIT's ORCD compute cluster, on a node with an NVIDIA L40S GPU - 46GB memory. The code should run on any machine with a CUDA-compatible GPU, and the performance difference between the CPU and GPU implementations may vary depending on the specific hardware used. Leveraging `KernelAbstractions.jl` one can also run the code on AMD/Intel/Apple GPUs with minimal change of code. 

### Introduction
For one that is not familiar with the usage of kernels for image editing, the idea is to apply a function (the kernel) to each pixel of the image, taking into account its neighbors. This is often used for tasks such as blurring, sharpening, edge detection, etc.

For example, if we want to apply a simple blur to an image, we can use a kernel that takes the average of the pixel and its neighbors. This can be done using a $3\times3$ kernel like this:

$$
\frac{1}{9}
\begin{bmatrix}
1 & 1 & 1 \\
1 & 1 & 1 \\
1 & 1 & 1 \\
\end{bmatrix}
$$

Similarly, for edge detection, we can use a kernel like this:
$$
\begin{bmatrix}
0 & -1 & 0 \\
-1 & 4 & -1 \\
0 & -1 & 0 \\
\end{bmatrix}
$$
The idea with the last one is that it will highlight the edges of the image by taking the difference between the pixel and its neighbors. If the pixel is similar to its neighbors, the result will be close to zero, but if it is different, the result will be larger.

For more examples of kernels for image editing, one can check out the [Wikipedia page](https://en.wikipedia.org/wiki/Kernel_(image_processing)) or the [Victor Powell's interactive page](https://setosa.io/ev/image-kernels/).

### Simple image kernels 

We'll first start with a simple example of applying a blur kernel, and a "sharpen" kernel to an image. We will use the `KernelAbstractions` library to do this in a fast and efficient way.

In [1]:
using KernelAbstractions, CUDA
using Images, FileIO

backend = CUDABackend() # Specify backend

CUDABackend(false, false)

In [ ]:
# Define helper functions to help translate images to tensors, and back.
function img_to_tensor(img)
    img_rgb = RGB.(img)
    tensor = channelview(img_rgb)
    return Float32.(tensor)
end

function tensor_to_img(tensor)
    @assert size(tensor, 1) == 3 "Tensor must have 3 channels"
    img = colorview(RGB, tensor)
    return img
end

Time to define some kernels!

In [ ]:
@kernel function blur_kernel!(output::AbstractArray, input::AbstractArray)
    k, i, j = @index(Global, NTuple)
    C, H, W = size(input)
    if (1 <=k <=C) && (2 <= i <= H-1) && (2 <= j <= W-1)
        output[k, i, j] = ((1/9) * (
            input[k,i,j] + 
            input[k,i-1,j] + 
            input[k,i,j-1] +
            input[k,i+1,j] +
            input[k,i,j+1] + 
            input[k,i+1,j+1] + 
            input[k,i-1,j+1] + 
            input[k,i-1,j-1] + 
            input[k,i+1,j-1]
            ))
    elseif (1 <=k <=C) && (1 <= i <= H) && (1 <= j <= W) #Edges
        output[k,i,j] = input[k,i,j]
    end
end

@kernel function edge_detection_kernel!(output::AbstractArray, input::AbstractArray)
    k, i, j = @index(Global, NTuple)
    C, H, W = size(input)
    if (1 <=k <=C) && (2 <= i <= H-1) && (2 <= j <= W-1)
        output[k, i, j] = abs(4*input[k,i,j]
        - input[k,i-1,j]
        - input[k,i,j-1]
        - input[k,i+1,j]
        - input[k,i,j+1])
    elseif (1 <=k <=C) && (1 <= i <= H) && (1 <= j <= W) #Edges
        output[k,i,j] = input[k,i,j]
    end
end


As has to be done with kernels, we have to define a "wrapper" function that will take care of the boundaries of the image, and specify the backend. 

In [ ]:
function blurImage!(output, img) 
    backend = KernelAbstractions.get_backend(img)
    kernel! = blur_kernel!(backend)
    kernel!(output, img, ndrange=size(img))
    KernelAbstractions.synchronize(backend)
    return output
end

function edgeDetectImage!(output, img) 
    backend = KernelAbstractions.get_backend(img)
    kernel! = edge_detection_kernel!(backend)
    kernel!(output, img, ndrange=size(img))
    KernelAbstractions.synchronize(backend)
    return output
end

Moreover, let's also define some versions of these functions that will copy the image (Note the lack of exclamation mark). While we are at it we can also define functions that operate directly on the `Images` file, and not the array (#heart multiple dispatch!).

In [ ]:
function blurImage(img::Array{Float32, 3})
    output = similar(img)
    imgC = deepcopy(img)
    blurImage!(output,imgC)
    return output
end

function edgeDetectImage(img::Array{Float32, 3})
    output = similar(img)
    imgC = deepcopy(img)
    edgeDetectImage!(output,imgC)
    return output
end

# The blurring after a single pass is not visible for the resolution 
# we are dealing with so we repeat it 30 times
function blurImage(img::Matrix{RGB{N0f8}}, passes=30)                                                                                                                                    
    img_tensor = img_to_tensor(img)                                                                                                                                                      
    output = similar(img_tensor)                                                                                                                                                         
    for _ in 1:passes                                                                                                                                                                    
        blurImage!(output, img_tensor)                                                                                                                                                   
        img_tensor .= output                                                                                                                                                             
    end
    return tensor_to_img(output)                                                                                                                                                         
end   

function edgeDetectImage(img::Matrix{RGB{N0f8}})
    img_tensor = img_to_tensor(img)
    output = similar(img_tensor)
    edgeDetectImage!(output,img_tensor)
    return tensor_to_img(output)
end


Now that we have all this boilerplate out of the way, we can apply our kernels to an image and see the results!
Check out this image of Nidarosdomen Cathedral in Trondheim, Norway. 


In [ ]:
# Load Image
original_dome = rotr90(load("nidarosdomen.jpg"))


Now look at the blurred version

In [ ]:
blurred_dome = blurImage(original_dome)

Cool! 

What about the edge detection?

In [ ]:
edges_dome = edgeDetectImage(original_dome)

Originally the plan was to edit an image of Phillip the Corgi, however, the lighting of the picture made it a bit unsuitable for this. Anyways, enjoy this unadulterated image of Phillip (credit to C. Wittens).

In [ ]:
original_phil = load("PhilC.JPG")

### Large scale editing - comparison with CPU
Of course, the above example is not very impressive, since the image is small and the kernels are simple. However, the real power of using kernels for image editing comes when we have many - or large - images.

Let us therefore attempt editing of the famous CIFAR10 dataset, which consists of 60,000 $32\times32$ images. We will apply the same edge detection kernel to the training set part of the images in the dataset (50k), and compare the performance of the GPU and CPU implementations.

To leverage the GPU-parallelism, we will edit the kernel so as to operate on a batch of images, instead of a single image. This is done by adding an extra dimension to the kernel.

In [ ]:
using BenchmarkTools, MLDatasets

In [ ]:
# As can be seen below, the CIFAR10 dataset contains images of dimension 32x32 
# - not huge in other words. 
dataset = CIFAR10()


In [ ]:
# See some of the images from the dataset
display(convert2image(dataset, 3142))
display(convert2image(dataset, 1618))
display(convert2image(dataset, 2718))

Now let's redefine the kernel to work on a batch of images.

In [ ]:
@kernel function edge_detection_kernel_batch!(output::AbstractArray, input::AbstractArray)
    b, k, i, j = @index(Global, NTuple)
    B, C, H, W = size(input)
    if (1 <= b <=B) && (1 <=k <=C) && (2 <= i <= H-1) && (2 <= j <= W-1)
        output[b, k, i, j] = abs(4*input[b,k,i,j]
        - input[b,k,i-1,j]
        - input[b,k,i,j-1]
        - input[b,k,i+1,j]
        - input[b,k,i,j+1])
    elseif (1 <= b <=B) && (1 <=k <=C) && (1 <= i <= H) && (1 <= j <= W) #Edges
        output[b,k,i,j] = input[b,k,i,j]
    end
end


function edgeDetectImageBatch!(output, img) 
    backend = KernelAbstractions.get_backend(img)
    kernel! = edge_detection_kernel_batch!(backend)
    kernel!(output, img, ndrange=size(img))
    KernelAbstractions.synchronize(backend)
    return output
end

In [ ]:
# Convert CIFAR10 dataset to standard array
batched_images = permutedims(dataset.features, (4, 3, 2, 1))
@show typeof(batched_images)
@show size(batched_images);

Now we are ready to apply the kernel to the entire dataset. We will time the execution of both the GPU and CPU implementations to see the performance difference.


To be fair (and conservative) we will include the time taken to transfer the data back and forth between the CPU and GPU in the timing of the GPU implementation.

In [ ]:
### CPU
edge_images_cpu = similar(batched_images);
@benchmark edgeDetectImageBatch!($edge_images_cpu, $batched_images)

In [ ]:
edge_images_gpu = similar(KernelAbstractions.adapt(backend, batched_images));

@benchmark begin
    images_gpu = KernelAbstractions.adapt(backend, batched_images)
    edge_images_gpu = similar(images_gpu)
    CUDA.@sync edgeDetectImageBatch!(edge_images_gpu, images_gpu)
    edge_images_gpu = Array(edge_images_gpu)
end

As we can see that GPU (with data transfer) faster by a factor $4$. Not insignificant, but not huge either. However, as we will see next, the bottleneck in this case is the data transfer, and if we were to keep the data on the GPU, we would see a much larger performance difference.

In [ ]:
edge_images_gpu = similar(KernelAbstractions.adapt(backend, batched_images));
images_gpu = KernelAbstractions.adapt(backend, batched_images)

@benchmark CUDA.@sync edgeDetectImageBatch!(edge_images_gpu, images_gpu)
# edge_images_gpu = Array(edge_images_gpu)

By excluding the data transfer time, we see that the GPU implementation is faster by a factor of 